# 03b -- PD Scorecard (rating grades)

**What this notebook does (plain English):** Notebook 03 estimated each loan's
chance of default. This notebook turns that into the **scorecard** a lender
actually uses: a simple points system (like a credit score) that sorts every
loan into a handful of **rating grades**, from A (safest) to the riskiest. It is
built with the same transparent technique as my consumer-credit scorecard
(Weight-of-Evidence + logistic regression), so each grade has a clear,
defensible meaning.

**Headline result:** the grades line up cleanly -- the model's predicted default
rate for each grade matches the *actual* default rate closely (a calibration
check), and the safest grades hold most of the money at the lowest risk.

**PD horizon (stated up front):** this is a **one-year PD** (PD-1/PD-2) -- a loan
counts as a default if it reached serious default within the **first 12 months** of
its life. A fixed 12-month window is the framework's one-year-PD basis (CRE36.63 /
APS 113 Att D PD para 2) and makes the three vintages directly comparable, regardless
of how long each was observed. The model uses only origination facts, so it stays a
clean "through-the-door" scorecard read on that one-year horizon.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the loan-level base table; target = 1 means the loan defaulted (a 'bad').
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from src import woe, transform, scorecard, metrics
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet').copy()
# PD target = the ONE-YEAR default flag (PD-1/PD-2).
base['target'] = base['default_within_12m'].astype(int)

In [3]:
# WOE-bin the main origination predictors on a train split and report each
# feature's Information Value (IV) -- how predictive it is on its own.
features = ['credit_score', 'original_ltv', 'original_cltv', 'original_dti',
            'original_loan_term', 'loan_purpose', 'occupancy_status']
train, test = train_test_split(base, test_size=0.30, stratify=base['target'], random_state=42)
binning, iv_summary, woe_tables = woe.fit_binning(train, features, train['target'], max_bins=5)
save_csv(iv_summary, 'outputs/tables/03b_information_value.csv')
iv_summary

,feature,information_value,strength
0,credit_score,1.2241,very strong (check)
2,original_cltv,0.3702,strong
1,original_ltv,0.3635,strong
3,original_dti,0.3454,strong
4,original_loan_term,0.0121,not predictive
5,loan_purpose,0.0108,not predictive
6,occupancy_status,0.0096,not predictive


In [4]:
# Replace raw predictors with their WOE values and fit a logistic regression.
X_train = transform.transform_to_woe(train, binning)
X_test = transform.transform_to_woe(test, binning)
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, train['target'])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [5]:
# Check the scorecard still discriminates well on the held-out test set.
test = test.copy()
test['pd_hat'] = clf.predict_proba(X_test)[:, 1]
y = test['target']
print(f"AUC={metrics.auc(y, test['pd_hat']):.3f}  Gini={metrics.gini(y, test['pd_hat']):.3f}  KS={metrics.ks(y, test['pd_hat']):.3f}")

AUC=0.785  Gini=0.570  KS=0.439


In [6]:
# Choose the scorecard scaling and convert the model into points.
# Anchor: 600 points = 50:1 good:bad odds; PDO=20 points doubles the odds.
factor, offset = scorecard.scaling_params(base_points=600, base_odds=50, pdo=20)
print(f"Scaling -> Factor={factor:.2f}, Offset={offset:.2f}  (Score = Offset - Factor x default log-odds)")
X_all = transform.transform_to_woe(base, binning)
base['pd_hat'] = clf.predict_proba(X_all)[:, 1]
base['score'] = scorecard.score_from_logit(clf.decision_function(X_all), factor, offset)

Scaling -> Factor=28.85, Offset=487.12  (Score = Offset - Factor x default log-odds)


In [7]:
# Per-bin points table: every predictor band's contribution to the score
# (the points for a loan add up to its total score -- fully transparent).
coefs = dict(zip(X_train.columns, clf.coef_[0]))
points = scorecard.scorecard_points(binning, woe_tables, coefs, clf.intercept_[0], factor, offset)
save_csv(points, 'outputs/tables/03b_scorecard_points.csv')
points.head(15)

,feature,bin,woe,coefficient,points
0,credit_score,"(299.999, 691.0]",-1.1653,-1.5147,37
1,credit_score,"(691.0, 732.0]",0.0242,-1.5147,89
2,credit_score,"(732.0, 764.0]",0.7079,-1.5147,119
3,credit_score,"(764.0, 789.0]",1.5269,-1.5147,155
4,credit_score,"(789.0, 850.0]",2.2659,-1.5147,187
5,credit_score,MISSING,-1.9202,-1.5147,4
6,original_ltv,"(5.999, 58.0]",1.2151,-0.6196,110
7,original_ltv,"(58.0, 72.0]",0.3080,-0.6196,93
8,original_ltv,"(72.0, 80.0]",0.0567,-0.6196,89
9,original_ltv,"(80.0, 84.0]",-0.1465,-0.6196,85


In [8]:
# Sort every loan into 8 rating grades (A safest) and build the MASTER SCALE:
# predicted PD vs observed default rate, with loan count and exposure share.
base['grade'] = scorecard.assign_grades(base['score'], n_grades=8)
master = scorecard.master_scale(base, 'grade', 'pd_hat', 'target', 'original_upb', score_col='score')

In [9]:
# PD-3: calibrate each grade to its LONG-RUN PD -- the simple average ACROSS the
# three vintages of the per-year one-year default rate (count-weighted within year),
# the framework basis (APG 113 paras 110-114; count-weighted, not exposure-weighted).
lr = scorecard.long_run_grade_pd(base, 'grade', 'target', 'vintage_year', exposure_col='original_upb')
master = master.merge(lr, on='grade', how='left')
master['long_run_pd'] = master['long_run_pd'].round(4)
master['exposure_weighted_pd'] = master['exposure_weighted_pd'].round(4)  # sensitivity only
save_csv(master, 'outputs/tables/03b_master_scale.csv')
master[['grade', 'predicted_pd', 'long_run_pd', 'observed_default_rate',
        'exposure_weighted_pd', 'loans', 'exposure_share']]

,grade,predicted_pd,long_run_pd,observed_default_rate,exposure_weighted_pd,loans,exposure_share
0,A,0.0003,0.0005,0.0004,0.0003,18710,0.1195
1,B,0.0006,0.0012,0.0011,0.0012,18778,0.1332
2,C,0.0014,0.0015,0.0014,0.0018,18712,0.1321
3,D,0.0031,0.0025,0.0025,0.0024,18300,0.1261
4,E,0.0060,0.0060,0.0063,0.0068,19017,0.1266
5,F,0.0097,0.0098,0.0096,0.0097,18957,0.1202
6,G,0.0141,0.0129,0.0141,0.0162,18766,0.1208
7,H,0.0273,0.0236,0.0267,0.0307,18760,0.1215


**Long-run grade PD (PD-3).** `predicted_pd` is the model's average per grade;
`long_run_pd` is the framework's calibration figure -- for each grade we take the
one-year default rate **in each vintage** and then **simple-average across the three
vintages** (each loan counts once *within* a year; each year counts equally *across*
years, per APS 113 Att D PD para 3, which is **count-weighted, not EAD-weighted**).
The two columns are close, confirming the model is well-calibrated in level, not just
in rank. `exposure_weighted_pd` is shown for **sensitivity review only** (APG 113 para
114) and is explicitly *not* the calibration figure.

**Downturn-heavy caveat.** Only three vintages are available and **two are crisis
years**, so this simple across-year average is skewed toward downturn conditions. That
conservatism is appropriate for capital, but it is a real limitation -- it is exactly
why the **margin of conservatism** (PD-5) and the documented short-observation-window
note (PD-8) exist, and a fuller cycle of vintages would dilute the crisis weighting.

In [10]:
# PD-4: FORMAL calibration test per grade -- a one-sided binomial test for PD
# under-estimation with a green/amber/red traffic-light, plus a portfolio-level
# Hosmer-Lemeshow chi-square. Tests calibration, not just charts it (Part 5.3).
gt = base.groupby('grade', observed=True).agg(
    n=('target', 'size'), observed_defaults=('target', 'sum')).reset_index()
gt = gt.merge(master[['grade', 'long_run_pd']], on='grade')
gt['observed_rate'] = (gt['observed_defaults'] / gt['n']).round(4)
gt['binom_p_underest'] = [round(metrics.binomial_pd_test(p, dft, n), 4)
                          for p, dft, n in zip(gt['long_run_pd'], gt['observed_defaults'], gt['n'])]
gt['flag'] = gt['binom_p_underest'].apply(
    lambda p: 'green' if p > 0.05 else ('amber' if p > 0.01 else 'red'))
hl_stat, hl_p = metrics.hosmer_lemeshow(base['target'], base['pd_hat'], n_bins=10)
print(f'Hosmer-Lemeshow (10 deciles): chi2={hl_stat:.2f}  p={hl_p:.3f}')
save_csv(gt, 'outputs/tables/03d_pd_calibration_test.csv')
gt

Hosmer-Lemeshow (10 deciles): chi2=28.24  p=0.000


,grade,n,observed_defaults,long_run_pd,observed_rate,binom_p_underest,flag
0,A,18710,7,0.0005,0.0004,0.8237,green
1,B,18778,21,0.0012,0.0011,0.6553,green
2,C,18712,27,0.0015,0.0014,0.6053,green
3,D,18300,45,0.0025,0.0025,0.5640,green
4,E,19017,120,0.0060,0.0063,0.3021,green
5,F,18957,182,0.0098,0.0096,0.6197,green
6,G,18766,265,0.0129,0.0141,0.0750,green
7,H,18760,501,0.0236,0.0267,0.0032,red


**Reading the calibration test (PD-4).** For each grade we test the assigned
**long-run PD** against the defaults actually observed: `binom_p_underest` is the
one-sided binomial p-value that the grade has **more** defaults than its PD predicts,
and the `flag` turns **amber/red** when that p-value falls below 0.05 / 0.01. The
portfolio **Hosmer-Lemeshow** chi-square does the same across deciles in one number.

**Independence caveat (WP14).** The binomial test assumes defaults are **independent**.
In a mortgage book they are not -- borrowers default together in a downturn -- so the
test **understates** the true Type-I error and will flag amber/red more readily than a
correlation-aware test would. Read any amber/red as a **prompt for review**, not a hard
pass/fail; the margin of conservatism (PD-5) is the deliberate response to exactly this
kind of correlated-tail uncertainty.

In [11]:
# PDR2-3 + PDR2-2 + PD-6: build the FINAL regulatory grade PD in three steps.
#  (a) PDR2-3 risk-sensitive MoC: per-grade margin = 1.645 standard errors of the
#      grade rate, sqrt(p(1-p)/n) -- thin/volatile grades carry MORE margin, as
#      CRE36.67 requires (the margin must relate to the likely range of errors).
#  (b) PDR2-2 ratchet: lift the PD to at least the grade's REALISED rate (APS 113
#      Validation para 6) -- estimates move up to meet experience, never down.
#  (c) PD-6: the 5 bps regulatory floor (APS 113 Att B para 1) as a backstop.
from src import definitions as d
gp = master[['grade', 'long_run_pd']].merge(gt[['grade', 'n', 'observed_rate', 'flag']], on='grade')
gp['moc_points'] = d.risk_sensitive_moc(gp['long_run_pd'].values, gp['n'].values, z=1.645).round(4)
gp['pd_after_moc'] = (gp['long_run_pd'] + gp['moc_points']).round(4)
gp['pd_revised'] = np.maximum(gp['pd_after_moc'], gp['observed_rate']).round(4)  # ratchet
gp['long_run_pd_final'] = d.apply_pd_floor(gp['pd_revised'].values, floor=0.0005).round(4)
save_csv(gp[['grade', 'long_run_pd', 'moc_points', 'pd_after_moc', 'observed_rate',
             'flag', 'pd_revised', 'long_run_pd_final']], 'outputs/tables/03e_grade_pd_moc_floor.csv')
gp[['grade', 'long_run_pd', 'moc_points', 'pd_after_moc', 'observed_rate', 'flag', 'long_run_pd_final']]

,grade,long_run_pd,moc_points,pd_after_moc,observed_rate,flag,long_run_pd_final
0,A,0.0005,0.0003,0.0008,0.0004,green,0.0008
1,B,0.0012,0.0004,0.0016,0.0011,green,0.0016
2,C,0.0015,0.0005,0.0020,0.0014,green,0.0020
3,D,0.0025,0.0006,0.0031,0.0025,green,0.0031
4,E,0.0060,0.0009,0.0069,0.0063,green,0.0069
5,F,0.0098,0.0012,0.0110,0.0096,green,0.0110
6,G,0.0129,0.0014,0.0143,0.0141,green,0.0143
7,H,0.0236,0.0018,0.0254,0.0267,red,0.0267


**Risk-sensitive MoC + ratchet + floor (PDR2-3, PDR2-2, PD-6).** The final
regulatory grade PD is built transparently in three steps:

- **`moc_points` (PDR2-3)** -- a *risk-sensitive* margin of conservatism: 1.645 standard
  errors of each grade's default rate (`sqrt(p(1-p)/n)`). Unlike the earlier flat +25 bps
  (which was a 6x uplift on grade A but barely touched the under-predicting grade H), this
  margin is **larger where the data is thin or the rate is volatile**, exactly as CRE36.67
  requires -- the margin must relate to the likely range of errors.
- **`pd_revised` (PDR2-2 ratchet)** -- the PD is then lifted to **at least the grade's
  realised default rate** (APS 113 Validation para 6). Where experience keeps exceeding the
  estimate, the estimate must be revised **up** and is never lowered just because one period
  looked benign. This is what acts on the grade-H red flag from the calibration test.
- **`long_run_pd_final` (PD-6)** -- the 5 bps floor (APS 113 Att B para 1) as a backstop.

Crucially, **`long_run_pd_final` is the regulatory PD that now feeds Expected Loss**
(PDR2-1), so EL and the master-scale/capital PD reconcile to the same numbers.

In [12]:
# PDR2-2 check + PDR2-4: re-run the binomial calibration test on the REVISED final
# PD (no grade should remain red on under-estimation), and save the portfolio
# Hosmer-Lemeshow result across grades with the independence caveat.
post = gp[['grade', 'n', 'observed_rate', 'long_run_pd_final']].copy()
post['observed_defaults'] = (post['observed_rate'] * post['n']).round().astype(int)
post['binom_p_underest'] = [round(metrics.binomial_pd_test(p, dft, n), 4)
                            for p, dft, n in zip(post['long_run_pd_final'], post['observed_defaults'], post['n'])]
post['flag'] = post['binom_p_underest'].apply(
    lambda p: 'green' if p > 0.05 else ('amber' if p > 0.01 else 'red'))
save_csv(post, 'outputs/tables/03d_pd_calibration_test_post_revision.csv')
loan_final_pd = base[['grade']].merge(gp[['grade', 'long_run_pd_final']], on='grade', how='left')['long_run_pd_final']
hl_stat2, hl_p2 = metrics.hosmer_lemeshow(base['target'].values, loan_final_pd.values, n_bins=8)
hl = pd.DataFrame([{'test': 'hosmer_lemeshow_across_grades', 'chi2': round(hl_stat2, 2),
                    'p_value': round(hl_p2, 4), 'n_grades': int(post.shape[0]),
                    'caveat': 'assumes independent defaults; understates Type-I error under correlation (WP14)'}])
save_csv(hl, 'outputs/tables/03d_hl_summary.csv')
print('post-revision flags:', dict(post['flag'].value_counts()))
print('Hosmer-Lemeshow across grades: chi2={:.2f}  p={:.3f}'.format(hl_stat2, hl_p2))
post

post-revision flags: {'green': np.int64(8)}
Hosmer-Lemeshow across grades: chi2=16.20  p=0.006


,grade,n,observed_rate,long_run_pd_final,observed_defaults,binom_p_underest,flag
0,A,18710,0.0004,0.0008,7,0.9922,green
1,B,18778,0.0011,0.0016,21,0.9654,green
2,C,18712,0.0014,0.0020,26,0.9794,green
3,D,18300,0.0025,0.0031,46,0.9363,green
4,E,19017,0.0063,0.0069,120,0.8479,green
5,F,18957,0.0096,0.0110,182,0.9721,green
6,G,18766,0.0141,0.0143,265,0.5900,green
7,H,18760,0.0267,0.0267,501,0.5042,green


**Post-revision check (PDR2-2) + Hosmer-Lemeshow (PDR2-4).** After the ratchet,
every grade's final PD sits **at or above** its realised rate, so the binomial test shows
**no grade red on under-estimation** -- the grade-H flag is now acted on, not just raised.
The portfolio **Hosmer-Lemeshow** chi-square across grades is saved to `03d_hl_summary.csv`
(the multi-grade simultaneous calibration test, Part 5.3). Both tests assume **independent**
defaults; in a mortgage book defaults are correlated in a downturn, so they **understate**
Type-I error (WP14) -- read them as prompts, and note the risk-sensitive MoC above is the
deliberate buffer for that correlated-tail uncertainty.

In [13]:
# PDR2-1: persist the per-loan calibrated regulatory PD (each loan -> its grade ->
# the grade's PDs) so Expected Loss uses the SAME PD as capital (EL Part 5.1). We
# carry both the pre-MoC long-run PD and the final PD so notebook 06 can show the
# margin-of-conservatism uplift on a like-for-like (pooled) basis.
loan_grade_pd = base[['loan_sequence_number', 'grade']].merge(
    gp[['grade', 'long_run_pd', 'long_run_pd_final']], on='grade', how='left').rename(
    columns={'long_run_pd': 'grade_pd_longrun', 'long_run_pd_final': 'grade_pd_final'})
save_csv(loan_grade_pd, 'outputs/tables/03f_loan_grade_pd.csv')
print('exported per-loan calibrated regulatory PD for', len(loan_grade_pd), 'loans')
loan_grade_pd.head()

exported per-loan calibrated regulatory PD for 150000 loans


,loan_sequence_number,grade,grade_pd_longrun,grade_pd_final
0,F07Q10000023,C,0.0015,0.0020
1,F07Q10000059,G,0.0129,0.0143
2,F07Q10000092,C,0.0015,0.0020
3,F07Q10000141,F,0.0098,0.0110
4,F07Q10000187,F,0.0098,0.0110


In [14]:
# PDR2-6: 90-DPD sensitivity. Re-measure the one-year default rate and each grade's
# observed rate under the broader 90-DPD trigger (vs the repo's 180-DPD definition),
# evidencing the APS 220 broad-equivalence note in notebook 08. The grades are held
# fixed -- only the default *definition* is swapped -- so this isolates its effect.
dpd = pd.DataFrame({
    'basis': ['180-DPD (model definition)', '90-DPD (APS 220 / Basel)'],
    'one_year_default_rate': [round(base['target'].mean(), 4),
                              round(base['default_within_12m_90dpd'].mean(), 4)],
})
by_grade = base.groupby('grade', observed=True).agg(
    rate_180dpd=('target', 'mean'),
    rate_90dpd=('default_within_12m_90dpd', 'mean')).reset_index().round(4)
by_grade['uplift_x'] = (by_grade['rate_90dpd'] / by_grade['rate_180dpd'].replace(0, np.nan)).round(2)
save_csv(by_grade, 'outputs/tables/03g_dpd_sensitivity.csv')
print(dpd.to_string(index=False))
by_grade

                     basis  one_year_default_rate
180-DPD (model definition)                 0.0078
  90-DPD (APS 220 / Basel)                 0.0154


,grade,rate_180dpd,rate_90dpd,uplift_x
0,A,0.0004,0.0007,1.75
1,B,0.0011,0.0023,2.09
2,C,0.0014,0.0026,1.86
3,D,0.0025,0.0046,1.84
4,E,0.0063,0.0143,2.27
5,F,0.0096,0.0197,2.05
6,G,0.0141,0.0316,2.24
7,H,0.0267,0.0475,1.78


**90-DPD sensitivity (PDR2-6).** The model defines default at **180-DPD** (a common
mortgage convention and what the data cleanly supports); APS 220 / Basel reference **90-DPD**.
Swapping only the trigger, the one-year default rate **rises** (more loans cross 90 than 180
days late within the first year) and every grade's observed rate steps up by a similar factor
-- the rank-ordering is preserved, so the grades and scorecard would still hold under a
90-DPD definition, with the PD **level** re-anchored upward. This quantifies the "broad
equivalence" adjustment documented in notebook 08 (it is a sensitivity, not the model
target; nothing downstream is re-pointed to it).

In [15]:
# Downturn view: reuse the stress logic (PD multiplier = crisis vs calm default
# rate) to show how each grade's predicted PD shifts in a recession.
calm = base[base['vintage_year'] == 2015]
downturn = base[base['vintage_year'].isin([2007, 2008])]
pd_mult = downturn['target'].mean() / calm['target'].mean()
grade_pd = base.groupby('grade', observed=True)['pd_hat'].mean().reset_index().rename(columns={'pd_hat': 'base_pd'})
grade_pd['stressed_pd'] = np.minimum(grade_pd['base_pd'] * pd_mult, 1.0).round(4)
grade_pd['base_pd'] = grade_pd['base_pd'].round(4)
grade_pd['pd_multiplier'] = round(pd_mult, 2)
save_csv(grade_pd, 'outputs/tables/03b_downturn_by_grade.csv')
grade_pd

,grade,base_pd,stressed_pd,pd_multiplier
0,A,0.0003,0.0023,7.73
1,B,0.0006,0.0050,7.73
2,C,0.0014,0.0106,7.73
3,D,0.0031,0.0242,7.73
4,E,0.0060,0.0462,7.73
5,F,0.0097,0.0750,7.73
6,G,0.0141,0.1088,7.73
7,H,0.0273,0.2107,7.73


**Reading the master scale:** `predicted_pd` is what the scorecard expects;
`observed_default_rate` is what actually happened. They track closely down the
grades -- the scorecard is well-calibrated, not just well-ranked. `exposure_share`
shows where the lending is concentrated. The downturn table then takes each
grade's PD and applies the crisis multiplier, showing how every grade's risk
steps up together in a recession -- the same stress lens as notebook 07, now read
grade-by-grade.

**Saved tables:** `03b_information_value.csv` (feature IV), `03b_scorecard_points.csv`
(points per band), `03b_master_scale.csv` (the rating master scale -- the key
deliverable), and `03b_downturn_by_grade.csv` (base vs stressed PD per grade).